# 旅行先レコメンドAI：データ作成とモデル構築

全国の市区町村の統計データ・土地利用データ・観光来訪者数を組み合わせ、旅行タイプに合う地域を推薦するためのデータを作ります。
最後に保存するCSVを、Streamlitアプリ（`app.py`）が読み込みます。

1. データを読み込む
2. 観光データを市区町村単位に集計する
3. 地域名の表記ゆれをそろえて結合する
4. 旅行向けの指標を作る
5. 観光来訪者数を予測するモデルを作る
6. 観光ポテンシャルと穴場候補を出す
7. 推薦関数を作って試す
8. アプリ用のCSVを保存する

In [9]:
# ライブラリ読み込み
import unicodedata
from pathlib import Path

import numpy as np
import pandas as pd
import optuna

from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

# 入力データの置き場所（場所を変えたらここだけ直す）
DATA_DIR = Path(".")

# アプリ用CSVの保存先（このノートブックと同じフォルダ）
OUTPUT_PATH = Path("travel_recommendation_data_ml.csv")

## 1. データを読み込む

In [11]:
# 統計＆土地利用データ
data = pd.read_csv(DATA_DIR / "CS2020_with_landuse_features.csv")

print(data.shape)
data.head()

(1728, 57)


,地域,TFR2022,R2020_log10(10万人あたり人口密度),J2020労働力人口女性割合,J2020第1次産業就業者比率【％】,T2020徒歩だけ,T2020鉄道・電車,T2020乗合バス,T2020勤め先・学校のバス,T2020自家用車,...,森林_ratio_valid,荒地_ratio_valid,建物用地_ratio_valid,道路_ratio_valid,鉄道_ratio_valid,その他用地_ratio_valid,河川湖沼_ratio_valid,海浜_ratio_valid,海水域_ratio_valid,ゴルフ場_ratio_valid
0,北海道 札幌市,1.09,3.652768,0.471028,0.47,0.111804,0.215397,0.048770,0.006697,0.351186,...,0.665860,0.027077,0.188809,0.015402,0.003091,0.044752,0.021985,0.000000,0.000000,0.004801
1,北海道 函館市,1.18,3.308714,0.470285,2.93,0.117959,0.020574,0.042005,0.009784,0.633064,...,0.832173,0.019015,0.073001,0.004246,0.001561,0.018422,0.003841,0.000500,0.002716,0.002997
2,北海道 小樽市,1.12,3.142921,0.475134,1.38,0.160906,0.062674,0.174740,0.010851,0.468862,...,0.765745,0.037571,0.105965,0.011579,0.003582,0.036571,0.004582,0.002249,0.004082,0.006165
3,北海道 旭川市,1.29,2.971137,0.466380,2.57,0.087152,0.005573,0.055984,0.010804,0.659623,...,0.586615,0.014077,0.096088,0.007610,0.002342,0.024242,0.032837,0.000000,0.000000,0.001969
4,北海道 室蘭市,1.38,3.278342,0.414711,0.93,0.153458,0.011764,0.064882,0.022295,0.663495,...,0.477144,0.037750,0.267334,0.016179,0.008731,0.123267,0.003852,0.001541,0.008988,0.006420


In [12]:
# 観光来訪者数データ（月別・市区町村別）
tourism = pd.read_csv(DATA_DIR / "city2022.csv", encoding="cp932")

print(tourism.shape)
tourism.head()

(22392, 9)


,年,月,地域区分,データ区分,都道府県コード,都道府県名,地域コード,地域名称,人数
0,2022,1,市区町村,観光来訪者数,1,北海道,1101,札幌市中央区,230903
1,2022,1,市区町村,観光来訪者数,1,北海道,1102,札幌市北区,12000
2,2022,1,市区町村,観光来訪者数,1,北海道,1103,札幌市東区,22333
3,2022,1,市区町村,観光来訪者数,1,北海道,1104,札幌市白石区,5575
4,2022,1,市区町村,観光来訪者数,1,北海道,1105,札幌市豊平区,22257


## 2. 観光データを市区町村単位に集計する

観光データは政令市が区ごとに分かれているため、「札幌市中央区 → 札幌市」のように市単位にまとめてから、月別の人数を年間で合計します。

In [13]:
def aggregate_city_name(name):
    # 例：札幌市中央区 → 札幌市、横浜市中区 → 横浜市
    if "市" in name and name.endswith("区"):
        return name.split("市")[0] + "市"
    return name

tourism["地域名称_集約"] = tourism["地域名称"].apply(aggregate_city_name)

tourism_year = (
    tourism.groupby(["都道府県名", "地域名称_集約"], as_index=False)["人数"].sum()
    .rename(columns={"人数": "年間観光来訪者数"})
)
tourism_year["地域"] = tourism_year["都道府県名"] + " " + tourism_year["地域名称_集約"]
tourism_year = tourism_year[["地域", "年間観光来訪者数"]]

print(tourism_year.shape)
tourism_year.head()

(1731, 2)


,地域,年間観光来訪者数
0,三重県 いなべ市,714717
1,三重県 亀山市,1098992
2,三重県 伊勢市,5486040
3,三重県 伊賀市,2210262
4,三重県 南伊勢町,44059


## 3. 地域名の表記ゆれをそろえて結合する

「鎌ケ谷市」と「鎌ヶ谷市」のように、統計データと観光データで地域名の表記が違うと結合できません。
以前は結合できなかった地域を来訪者数0として扱っていたため、観光人気度0の地域が穴場ランキングの上位に並んでいました。
ここでは表記をそろえたうえで、結合できない地域を一覧で確認します。

In [14]:
def normalize_name(name):
    name = unicodedata.normalize("NFKC", str(name))  # 全角・半角をそろえる
    name = name.replace("ヶ", "ケ")                   # 小さい「ヶ」を「ケ」にそろえる
    return name

data["地域"] = data["地域"].map(normalize_name)
tourism_year["地域"] = tourism_year["地域"].map(normalize_name)

# 下の確認で見つかった字の違いを追加する（観光データ側の名前 → 統計データ側の名前）
# 例：{"宮城県 塩釜市": "宮城県 塩竈市"}
name_fix = {}
tourism_year["地域"] = tourism_year["地域"].replace(name_fix)
tourism_year = tourism_year.groupby("地域", as_index=False)["年間観光来訪者数"].sum()

# 結合できない地域を確認する
check = data[["地域"]].merge(tourism_year, on="地域", how="left", indicator=True)
missing = check.loc[check["_merge"] == "left_only", "地域"].tolist()

print("結合できなかった地域数:", len(missing))
print(missing)

結合できなかった地域数: 11
['北海道 今金町', '北海道 古平町', '北海道 上砂川町', '北海道 猿払村', '北海道 訓子府町', '宮城県 塩竈市', '東京都 御蔵島村', '新潟県 粟島浦村', '長野県 北相木村', '大阪府 河内長野市', '高知県 檮原町']


In [15]:
# 結合できなかった地域について、観光データ側に似た名前がないか探す
# 似た名前が見つかったら上の name_fix に追加し、上のセルからやり直す
for name in missing:
    pref, city = name.split(" ", 1)
    hits = tourism_year[
        tourism_year["地域"].str.startswith(pref)
        & tourism_year["地域"].str.contains(city[0], regex=False)
    ]["地域"].tolist()
    print(name, "→", hits if hits else "観光データになし")

北海道 今金町 → 観光データになし
北海道 古平町 → ['北海道 木古内町']
北海道 上砂川町 → ['北海道 上ノ国町', '北海道 上士幌町', '北海道 上富良野町', '北海道 上川町', '北海道 滝上町']
北海道 猿払村 → 観光データになし
北海道 訓子府町 → 観光データになし
宮城県 塩竈市 → ['宮城県 塩竃市']
東京都 御蔵島村 → 観光データになし
新潟県 粟島浦村 → 観光データになし
長野県 北相木村 → ['長野県 筑北村']
大阪府 河内長野市 → ['大阪府 河南町']
高知県 檮原町 → 観光データになし


In [16]:
# 結合する（以前のデータに観光関連の列があれば消してから）
drop_cols = [c for c in data.columns if "観光来訪者数" in c or "観光人気度" in c]
data = data.drop(columns=drop_cols, errors="ignore")

data = data.merge(tourism_year, on="地域", how="left")

# 観光データがない地域は0人にせず、「データなし」として区別する
data["観光データあり"] = data["年間観光来訪者数"].notna()

print("観光データあり:", data["観光データあり"].sum(), "地域")
print("観光データなし:", (~data["観光データあり"]).sum(), "地域")

観光データあり: 1717 地域
観光データなし: 11 地域


## 4. 旅行向けの指標を作る

各データを0〜100に換算し、重みを付けて旅行者に分かる指標にします。重みは自分の判断で決めています。

In [17]:
def minmax(s):
    if s.max() == s.min():
        return s * 0
    return (s - s.min()) / (s.max() - s.min())

# 観光人気度（実績）：来訪者数は桁が大きく違うためlog10をとってから0〜100にする
# 観光データがない地域は空欄のまま
data["log_年間観光来訪者数"] = np.log10(data["年間観光来訪者数"] + 1)
data["観光人気度"] = minmax(data["log_年間観光来訪者数"]) * 100

data.sort_values("年間観光来訪者数", ascending=False)[
    ["地域", "年間観光来訪者数", "観光人気度"]
].head(10)

,地域,年間観光来訪者数,観光人気度
973,愛知県 名古屋市,55145992.0,100.000000
1101,大阪府 大阪市,34146093.0,96.591824
616,東京都 千代田区,30760962.0,95.849506
1075,京都府 京都市,27879311.0,95.150136
618,東京都 港区,22475451.0,93.618163
619,東京都 新宿区,22221546.0,93.537382
631,東京都 豊島区,21668902.0,93.358317
628,東京都 渋谷区,19852443.0,92.735811
617,東京都 中央区,19206377.0,92.500572
678,神奈川県 横浜市,19015393.0,92.429516


In [18]:
# 基本の特徴量
pop_density = data["R2020_log10(10万人あたり人口密度)"]

building = data["建物用地_ratio_valid"]
road = data["道路_ratio_valid"]
rail_land = data["鉄道_ratio_valid"]

forest = data["森林_ratio_valid"]
rice = data["田_ratio_valid"]
farm = data["その他農用地_ratio_valid"]
river = data["河川湖沼_ratio_valid"]
beach = data["海浜_ratio_valid"]
sea = data["海水域_ratio_valid"]

walk = data["T2020徒歩だけ"]
train = data["T2020鉄道・電車"]
car = data["T2020自家用車"]

primary = data["J2020第1次産業就業者比率【％】"]

hotel_food = data["S2021_従業者数（民営）（宿泊業、飲食サービス業）"]
retail = data["S2021従業者数（民営）（小売業）"]
entertainment = data["S2021_従業者数（民営）（生活関連サービス業、娯楽業"]

In [19]:
# 旅行向けの指標（0〜100）
data["都会度"] = (
    0.35 * minmax(pop_density)
    + 0.30 * minmax(building)
    + 0.15 * minmax(road)
    + 0.20 * minmax(train)
) * 100

data["自然度"] = (
    0.45 * minmax(forest)
    + 0.25 * minmax(rice + farm)
    + 0.15 * minmax(river)
    + 0.15 * (1 - minmax(building))
) * 100

data["海度"] = (
    0.70 * minmax(sea)
    + 0.30 * minmax(beach)
) * 100

data["山っぽさ"] = (
    0.70 * minmax(forest)
    + 0.20 * (1 - minmax(building))
    + 0.10 * (1 - minmax(pop_density))
) * 100

data["田舎度"] = (
    0.30 * minmax(forest)
    + 0.25 * minmax(rice + farm)
    + 0.20 * minmax(car)
    + 0.15 * minmax(primary)
    + 0.10 * (1 - minmax(pop_density))
) * 100

data["電車旅向き"] = (
    0.55 * minmax(train)
    + 0.20 * minmax(rail_land)
    + 0.15 * minmax(walk)
    + 0.10 * minmax(pop_density)
) * 100

data["車旅向き"] = (
    0.55 * minmax(car)
    + 0.20 * minmax(road)
    + 0.15 * (1 - minmax(train))
    + 0.10 * (1 - minmax(pop_density))
) * 100

data["観光・街歩き向き"] = (
    0.30 * minmax(hotel_food)
    + 0.25 * minmax(retail)
    + 0.20 * minmax(entertainment)
    + 0.15 * minmax(walk)
    + 0.10 * minmax(train)
) * 100

score_cols = ["都会度", "自然度", "海度", "山っぽさ", "田舎度", "電車旅向き", "車旅向き", "観光・街歩き向き"]
data[["地域"] + score_cols].head()

,地域,都会度,自然度,海度,山っぽさ,田舎度,電車旅向き,車旅向き,観光・街歩き向き
0,北海道 札幌市,41.894755,43.859379,0.000000,65.357749,30.861230,30.530268,34.232558,19.059049
1,北海道 函館市,27.445808,53.257512,2.753736,80.739575,44.343092,12.390420,56.563040,19.077981
2,北海道 小樽市,28.739773,49.095775,5.045545,75.782821,38.111970,17.302367,46.428972,19.714833
3,北海道 旭川市,24.504167,48.408605,0.000000,63.762719,44.328090,9.260559,59.962465,15.762068
4,北海道 室蘭市,34.186023,34.056736,9.043601,51.321299,34.136245,13.975297,60.070615,15.963786


## 5. 観光来訪者数を予測するモデルを作る

地域の特徴から、年間観光来訪者数（log10）をランダムフォレストで予測します。
学習には観光データがある地域だけを使います。ハイパーパラメータはOptunaで調整します。

In [20]:
feature_cols = score_cols + [
    "R2020_log10(10万人あたり人口密度)",
    "建物用地_ratio_valid",
    "道路_ratio_valid",
    "鉄道_ratio_valid",
    "森林_ratio_valid",
    "田_ratio_valid",
    "その他農用地_ratio_valid",
    "河川湖沼_ratio_valid",
    "海浜_ratio_valid",
    "海水域_ratio_valid",
    "S2021_従業者数（民営）（宿泊業、飲食サービス業）",
    "S2021従業者数（民営）（小売業）",
    "S2021_従業者数（民営）（生活関連サービス業、娯楽業",
]

train_data = data[data["観光データあり"]]
X = train_data[feature_cols]
y = train_data["log_年間観光来訪者数"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("学習:", X_train.shape, " テスト:", X_test.shape)

学習: (1373, 21)  テスト: (344, 21)


In [21]:
def objective(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 200, 1000),
        "max_depth": trial.suggest_int("max_depth", 3, 30),
        "min_samples_split": trial.suggest_int("min_samples_split", 2, 20),
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 10),
        "max_features": trial.suggest_categorical("max_features", ["sqrt", "log2", None]),
        "random_state": 42,
        "n_jobs": -1,
    }
    model = RandomForestRegressor(**params)
    cv = KFold(n_splits=5, shuffle=True, random_state=42)
    scores = cross_val_score(
        model, X_train, y_train, cv=cv,
        scoring="neg_root_mean_squared_error", n_jobs=-1,
    )
    return -scores.mean()

optuna.logging.set_verbosity(optuna.logging.WARNING)
study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=30)

print("Best CV RMSE:", study.best_value)
print("Best params:", study.best_params)

/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/sklearn/utils/parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/sklearn/utils/parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/sklearn/utils/parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warn

Best CV RMSE: 0.5886257735390763
Best params: {'n_estimators': 616, 'max_depth': 24, 'min_samples_split': 7, 'min_samples_leaf': 1, 'max_features': 'log2'}


In [22]:
rf_best = RandomForestRegressor(**study.best_params, random_state=42, n_jobs=-1)
rf_best.fit(X_train, y_train)

y_pred_train = rf_best.predict(X_train)
y_pred_test = rf_best.predict(X_test)

print("Train R2:", r2_score(y_train, y_pred_train))
print("Test R2:", r2_score(y_test, y_pred_test))
print("Test MAE:", mean_absolute_error(y_test, y_pred_test))
print("Test RMSE:", np.sqrt(mean_squared_error(y_test, y_pred_test)))

Train R2: 0.8389647882969238
Test R2: 0.43584402421172974
Test MAE: 0.46015508709530056
Test RMSE: 0.6079737149252986


## 6. 観光ポテンシャルと穴場候補を出す

- ML観光ポテンシャル：地域の特徴から予測した来訪者数を0〜100にしたもの（全地域）
- 観光ポテンシャル差：予測 − 実績。大きいほど「特徴の割に人が少ない」穴場の候補（観光データがある地域のみ）

In [23]:
# 観光データがない地域も含めて、全地域について予測する
data["ML予測_log観光来訪者数"] = rf_best.predict(data[feature_cols])
data["ML観光ポテンシャル"] = minmax(data["ML予測_log観光来訪者数"]) * 100

data["観光ポテンシャル差"] = data["ML観光ポテンシャル"] - data["観光人気度"]

In [24]:
# 穴場候補ランキング（観光データがある地域のみ）
data[data["観光データあり"]].sort_values("観光ポテンシャル差", ascending=False)[
    ["地域", "観光人気度", "ML観光ポテンシャル", "観光ポテンシャル差"] + score_cols
].head(20)

,地域,観光人気度,ML観光ポテンシャル,観光ポテンシャル差,都会度,自然度,海度,山っぽさ,田舎度,電車旅向き,車旅向き,観光・街歩き向き
167,北海道 浜中町,7.117128,35.585462,28.468334,5.159430,49.103492,0.000000,52.159084,62.522157,6.808483,65.176581,13.707598
1707,沖縄県 嘉手納町,34.968778,62.433184,27.464406,28.231165,26.388662,0.000000,39.464415,30.145285,8.644754,65.180872,16.126970
128,北海道 滝上町,12.956897,38.502371,25.545474,4.432471,58.915206,0.000000,93.876669,58.708426,5.449249,66.813094,15.324563
1452,高知県 三原村,21.631409,43.514351,21.882942,12.157230,59.012547,0.000000,91.908565,58.195368,5.787233,69.331018,11.914802
389,福島県 浅川町,30.081355,50.062640,19.981285,18.725673,50.326351,0.000000,64.447977,53.712790,9.572430,71.197110,11.123290
303,秋田県 八郎潟町,31.937379,51.683620,19.746241,23.101634,41.661651,0.000000,37.919771,47.050897,11.451733,66.329523,15.156465
1553,長崎県 佐々町,38.800447,57.383042,18.582595,25.686171,47.979336,1.490246,64.851589,47.668656,11.144090,67.004622,16.973883
868,長野県 喬木村,32.264946,50.408425,18.143479,18.845714,56.278068,0.000000,83.059438,56.097849,6.979255,72.027369,11.010625
1138,大阪府 熊取町,45.279109,62.832599,17.553490,44.827424,28.908027,0.000000,38.790907,26.263649,21.158847,40.824366,14.625248
634,東京都 板橋区,65.452920,82.105245,16.652326,79.846319,4.748756,0.000000,4.282796,1.502386,61.716704,7.596379,22.261616


## 7. 推薦関数を作って試す

In [25]:
def recommend_area_ml(
    data,
    urban=0, nature=0, sea=0, mountain=0, rural=0,
    train_trip=0, car_trip=0, sightseeing=0,
    ml_tourism=0.5,
    top_n=10,
):
    total_weight = (
        urban + nature + sea + mountain + rural
        + train_trip + car_trip + sightseeing + ml_tourism
    )
    if total_weight == 0:
        raise ValueError("少なくとも1つの条件に重みを入れてください")

    result = data.copy()
    result["おすすめスコア"] = (
        urban * result["都会度"]
        + nature * result["自然度"]
        + sea * result["海度"]
        + mountain * result["山っぽさ"]
        + rural * result["田舎度"]
        + train_trip * result["電車旅向き"]
        + car_trip * result["車旅向き"]
        + sightseeing * result["観光・街歩き向き"]
        + ml_tourism * result["ML観光ポテンシャル"]
    ) / total_weight

    display_cols = ["地域", "おすすめスコア", "観光人気度", "ML観光ポテンシャル", "観光ポテンシャル差"] + score_cols
    return result.sort_values("おすすめスコア", ascending=False)[display_cols].head(top_n)

In [26]:
# 旅行タイプのプリセット
travel_patterns_ml = {
    "海と自然でのんびり": {"sea": 1.0, "nature": 0.8, "ml_tourism": 0.6},
    "海沿い観光・リゾート": {"sea": 1.0, "sightseeing": 0.8, "ml_tourism": 0.7},
    "山と自然を楽しむ": {"mountain": 1.0, "nature": 1.0, "rural": 0.5, "ml_tourism": 0.6},
    "田舎でのんびり": {"rural": 1.0, "nature": 0.8, "car_trip": 0.6, "ml_tourism": 0.4},
    "都会で街歩き": {"urban": 1.0, "sightseeing": 0.8, "train_trip": 0.6, "ml_tourism": 0.5},
    "電車で行ける観光地": {"train_trip": 1.0, "sightseeing": 0.8, "urban": 0.4, "ml_tourism": 0.6},
}

def recommend_by_pattern_ml(data, pattern_name, top_n=20):
    if pattern_name not in travel_patterns_ml:
        raise ValueError(f"使える旅行タイプ: {list(travel_patterns_ml.keys())}")
    return recommend_area_ml(data, top_n=top_n, **travel_patterns_ml[pattern_name])

In [27]:
recommend_by_pattern_ml(data, "海と自然でのんびり", top_n=10)

,地域,おすすめスコア,観光人気度,ML観光ポテンシャル,観光ポテンシャル差,都会度,自然度,海度,山っぽさ,田舎度,電車旅向き,車旅向き,観光・街歩き向き
1715,沖縄県 座間味村,51.886894,48.822366,22.351044,-26.471322,15.646162,49.765015,71.305906,79.023367,34.783201,19.068919,35.183567,46.431111
677,東京都 小笠原村,49.681251,43.613296,14.182340,-29.430955,7.895623,50.906998,70.000000,83.506251,39.540315,14.819389,38.547114,32.110332
1717,沖縄県 渡名喜村,48.559760,20.786052,14.527432,-6.258620,16.044251,46.545180,70.590820,71.718261,36.971794,17.526315,38.631116,35.748825
1714,沖縄県 渡嘉敷村,46.358922,41.023233,23.825084,-17.198149,16.956179,51.460065,55.798309,81.379837,40.513268,15.302928,47.738504,40.291104
1394,香川県 直島町,46.315367,58.636209,25.400305,-33.235904,22.905439,38.554626,65.072997,60.254601,31.486121,8.911615,45.414110,15.602115
1248,和歌山県 太地町,45.790406,62.357067,48.876708,-13.480359,27.819326,44.059684,45.323202,67.952244,43.599677,10.498559,64.026717,18.093428
1552,長崎県 小値賀町,44.932510,48.029694,24.332778,-23.696917,15.375095,45.923726,56.499377,61.463968,48.796648,9.301229,57.202855,25.468232
251,宮城県 塩竈市,42.778683,NaN,52.329542,NaN,51.154719,16.086690,58.401762,25.094669,22.959432,25.183021,53.760720,16.829282
1036,三重県 鳥羽市,42.420800,77.248024,71.367047,-5.880977,21.103392,53.757524,15.983673,82.447658,47.779130,12.470878,56.712844,29.347373
1039,三重県 志摩市,42.044901,73.919106,68.164549,-5.754558,22.833948,48.670148,21.072914,72.399656,48.606163,9.224435,67.371190,21.977681


「都会で街歩き」は、そのままだと東京23区が上位を独占します。区は人口密度などが突出しているためです。
アプリでは区を除外する選択肢を用意しています。

In [28]:
recommend_by_pattern_ml(data, "都会で街歩き", top_n=10)

,地域,おすすめスコア,観光人気度,ML観光ポテンシャル,観光ポテンシャル差,都会度,自然度,海度,山っぽさ,田舎度,電車旅向き,車旅向き,観光・街歩き向き
631,東京都 豊島区,68.554260,93.358317,91.516558,-1.841759,87.040325,1.256182,0.000000,1.714226,0.159573,75.271087,5.669264,26.057621
621,東京都 台東区,65.850903,90.932201,100.000000,9.067799,83.081530,4.676194,0.000000,5.267706,0.824416,66.170925,11.862631,22.729419
619,東京都 新宿区,65.837683,93.537382,97.416324,3.878942,84.233339,3.252535,0.000000,4.507652,1.145125,67.752706,9.479467,21.670196
624,東京都 品川区,65.736377,81.599535,88.165194,6.565659,80.128374,5.177418,5.007457,7.004974,0.994692,85.463277,8.053139,18.933195
622,東京都 墨田区,65.555031,84.899140,97.679510,12.780370,82.729353,6.102775,0.000000,4.503391,0.642696,67.071512,11.282865,22.871969
620,東京都 文京区,65.209501,87.640814,95.514485,7.873671,85.245816,3.795310,0.000000,5.216957,1.375475,67.666603,8.660916,19.380667
628,東京都 渋谷区,64.148897,92.735811,94.067699,1.331888,79.892880,6.448476,0.000000,9.945034,3.581521,67.506310,8.105951,23.251608
633,東京都 荒川区,63.161556,73.030480,81.602076,8.571596,79.249681,4.299360,0.000000,3.576983,0.495341,75.081958,4.909424,22.585774
629,東京都 中野区,63.131307,77.574766,81.921141,4.346375,84.121688,0.705612,0.000000,0.516878,0.369492,65.954137,2.437615,23.032562
632,東京都 北区,62.846927,71.285963,81.719100,10.433138,78.614271,7.170789,0.000000,5.924585,1.087344,73.063529,6.334278,23.680187


In [29]:
data_no_ku = data[~data["地域"].str.endswith("区")]
recommend_by_pattern_ml(data_no_ku, "都会で街歩き", top_n=10)

,地域,おすすめスコア,観光人気度,ML観光ポテンシャル,観光ポテンシャル差,都会度,自然度,海度,山っぽさ,田舎度,電車旅向き,車旅向き,観光・街歩き向き
641,東京都 武蔵野市,57.915197,73.186893,68.407818,-4.779075,76.369572,2.168664,0.000000,3.290575,1.911392,58.259783,6.496057,28.030901
679,神奈川県 川崎市,57.832086,89.070458,92.522874,3.452416,71.588598,8.106835,6.188044,9.628527,4.473478,55.295058,11.789775,20.857473
1101,大阪府 大阪市,56.848697,96.591824,98.100598,1.508774,71.728679,7.458713,2.992677,6.648348,2.458827,49.684247,17.478614,17.839619
1106,大阪府 吹田市,55.835697,78.150248,85.029727,6.879479,71.917950,8.250151,0.000000,12.035592,5.582625,52.297123,20.227644,20.140541
586,千葉県 浦安市,55.775072,89.067094,88.994303,-0.072791,68.609380,6.766621,11.919997,8.432870,2.661598,41.661213,22.427893,29.555561
678,神奈川県 横浜市,55.192666,92.429516,88.257356,-4.172160,69.307319,10.434729,2.762077,13.113854,8.206345,49.775105,17.155889,20.947089
646,東京都 調布市,54.808397,72.127385,69.847213,-2.280171,72.965655,7.045107,0.000000,7.021336,3.818078,52.691707,10.210011,24.300081
564,千葉県 市川市,53.428678,70.775601,69.933739,-0.841862,69.515601,10.365815,2.380854,8.816729,6.852105,53.217545,14.586093,23.162713
518,埼玉県 蕨市,53.095075,51.432655,53.970849,2.538194,77.546174,0.259846,0.000000,0.994944,2.515007,50.188537,11.762708,24.163748
655,東京都 狛江市,53.092019,54.583821,52.548305,-2.035516,75.488416,3.856030,0.000000,2.257912,2.135794,54.772833,7.069253,24.175733


## 8. アプリ用のCSVを保存する

In [30]:
data.to_csv(OUTPUT_PATH, index=False, encoding="utf-8-sig")
print("保存しました:", OUTPUT_PATH.resolve())

保存しました: /Users/miyazakikaito/Desktop/趣味/trip/travel_recommendation_data_ml.csv
